**Q1**

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re

In [3]:
base_url = "https://books.toscrape.com/"

# fetch the homepage
response = requests.get(base_url + "index.html")
soup = BeautifulSoup(response.text, "html.parser")

target_categories = ["Travel", "Romance", "Religion", "Poetry"]

category_links = {}

sidebar = soup.find('div', class_='side_categories')
category_tags = sidebar.find_all('a')

for tag in category_tags:
    cat_name = tag.text.strip()
    if cat_name in target_categories:
        link = tag['href']
        category_links[cat_name] = base_url + link

print("found the following category links:")
for name, link in category_links.items():
    print(f"{name}: {link}")

found the following category links:
Travel: https://books.toscrape.com/catalogue/category/books/travel_2/index.html
Romance: https://books.toscrape.com/catalogue/category/books/romance_8/index.html
Religion: https://books.toscrape.com/catalogue/category/books/religion_12/index.html
Poetry: https://books.toscrape.com/catalogue/category/books/poetry_23/index.html


In [5]:
# helper function to convert text star ratings to numbers
def convert_rating(class_list):
    mapping = {'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5}
    for cls in class_list:
        if cls in mapping:
            return mapping[cls]
    return 0

scraped_books = {} # key: book title, value: dictionary of book details

for cat_name, cat_url in category_links.items():
    print(f"\n Scraping Category: {cat_name} ")
    current_page_url = cat_url

    actual_books_scraped = 0
    expected_total = 0

    while True:
        # fetch the current listing page
        response = requests.get(current_page_url)
        soup = BeautifulSoup(response.text, "html.parser")

        #expected total results count
        if expected_total == 0:
            total_tag = soup.select_one('.form-horizontal strong')
            if total_tag:
                expected_total = int(total_tag.text)
                print(f"expected total books in {cat_name}: {expected_total}")

        #extract all books on the current page
        books_on_page = soup.find_all('article', class_='product_pod')

        for book in books_on_page:

            title = book.find('h3').find('a')['title'].strip()

            if title in scraped_books:
                actual_books_scraped += 1
                continue

            # construct the absolute URL for the individual product page
            book_rel_link = book.find('h3').find('a')['href']
            clean_link = book_rel_link.replace('../../../', '')
            book_url = f"https://books.toscrape.com/catalogue/{clean_link}"

            book_resp = requests.get(book_url)
            book_soup = BeautifulSoup(book_resp.text, "html.parser")

            price = book_soup.find('p', class_='price_color').text.strip()

            availability = book_soup.find('p', class_='instock availability').text.strip()

            star_tag = book_soup.find('p', class_='star-rating')
            star_rating = convert_rating(star_tag['class']) if star_tag else 0

            upc = ""
            upc_th = book_soup.find('th', string='UPC')
            if upc_th:
                upc = upc_th.find_next_sibling('td').text.strip()

            description = ""
            desc_div = book_soup.find('div', id='product_description')
            if desc_div:
                description = desc_div.find_next_sibling('p').text.strip()


            scraped_books[title] = {
                'Title': title,
                'Price': price,
                'In-stock availability text': availability,
                'Star rating': star_rating,
                'UPC': upc,
                'Product description': description,
                'Category': cat_name
            }
            actual_books_scraped += 1

        #handle pagination
        next_btn = soup.find('li', class_='next')
        if next_btn:
            next_page_rel = next_btn.find('a')['href']
            current_page_url = current_page_url.rsplit('/', 1)[0] + '/' + next_page_rel
        else:
            break

    # comparing expected vs actual
    print(f"Validation for {cat_name}:")
    if actual_books_scraped == expected_total:
        print(f"PASS ^^ : Scraped {actual_books_scraped} books, exactly matching the expected {expected_total}.")
    else:
        print(f"FAIL :( : Scraped {actual_books_scraped} books, but expected {expected_total}.")


 Scraping Category: Travel 
expected total books in Travel: 11
Validation for Travel:
PASS ^^ : Scraped 11 books, exactly matching the expected 11.

 Scraping Category: Romance 
expected total books in Romance: 35
Validation for Romance:
PASS ^^ : Scraped 35 books, exactly matching the expected 35.

 Scraping Category: Religion 
expected total books in Religion: 7
Validation for Religion:
PASS ^^ : Scraped 7 books, exactly matching the expected 7.

 Scraping Category: Poetry 
expected total books in Poetry: 19
Validation for Poetry:
PASS ^^ : Scraped 19 books, exactly matching the expected 19.


In [6]:
books_list = list(scraped_books.values())

df_static = pd.DataFrame(books_list)

csv_filename = '23L_2503_versionB_static_books.csv'
df_static.to_csv(csv_filename, index=False)

print(f"Data saved to {csv_filename}")

df_static.head()

Data saved to 23L_2503_versionB_static_books.csv


,Title,Price,In-stock availability text,Star rating,UPC,Product description,Category
0,It's Only the Himalayas,Â£45.17,In stock (19 available),2,a22124811bfa8350,"âWherever you go, whatever you do, just . . ...",Travel
1,Full Moon over Noahâs Ark: An Odyssey to Mou...,Â£49.43,In stock (15 available),4,ce60436f52c5ee68,Acclaimed travel writer Rick Antonson sets his...,Travel
2,See America: A Celebration of Our National Par...,Â£48.87,In stock (14 available),3,f9705c362f070608,To coincide with the 2016 centennial anniversa...,Travel
3,Vagabonding: An Uncommon Guide to the Art of L...,Â£36.94,In stock (8 available),2,1809259a5a5f1d8d,With a new foreword by Tim Ferriss â¢Thereâ...,Travel
4,Under the Tuscan Sun,Â£37.33,In stock (7 available),3,a94350ee74deaa07,A CLASSIC FROM THE BESTSELLING AUTHOR OF UNDER...,Travel


**Q2**

In [7]:
test_resp = requests.get("https://quotes.toscrape.com/scroll")
test_soup = BeautifulSoup(test_resp.text, "html.parser")
quotes_found = test_soup.find_all('div', class_='quote')
print(f"quotes found using plain requests: {len(quotes_found)}")

quotes found using plain requests: 0


In [10]:
# Install Google Chrome
!wget https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!dpkg -i google-chrome-stable_current_amd64.deb
!apt-get -f install -y

# Install Selenium and ChromeDriver Manager
!pip install selenium webdriver-manager

--2026-09-12 16:11:38--  https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
Resolving dl.google.com (dl.google.com)... 142.250.152.93, 142.250.152.136, 142.250.152.190, ...
Connecting to dl.google.com (dl.google.com)|142.250.152.93|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 141931876 (135M) [application/x-debian-package]
Saving to: ‘google-chrome-stable_current_amd64.deb’

google-chrome-stabl 100%[===================>] 135.36M   137MB/s    in 1.0s    

2026-09-12 16:11:39 (137 MB/s) - ‘google-chrome-stable_current_amd64.deb’ saved [141931876/141931876]

Selecting previously unselected package google-chrome-stable.
(Reading database ... 122797 files and directories currently installed.)
Preparing to unpack google-chrome-stable_current_amd64.deb ...
Unpacking google-chrome-stable (153.0.8010.36-1) ...
dpkg: dependency problems prevent configuration of google-chrome-stable:
 google-chrome-stable depends on libatk-bridge2.0-0 (>= 2.5.3)

In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
import pandas as pd

In [2]:
options = webdriver.ChromeOptions()
options.add_argument('--headless')
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')

# Set up the ChromeDriver service
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=options)

url = "https://quotes.toscrape.com/scroll"
driver.get(url)

scraped_quotes = {} # key: quote text, value: quote data
current_batch = 0
empty_scroll_count = 0

wait = WebDriverWait(driver, 5)#explicit wait (up to 5 seconds maximum per scroll)

print("starting dynamic scrolling..")

while True:

    quote_elements = driver.find_elements(By.CLASS_NAME, "quote") #grab all quote elements currently loaded in the DOM

    # extract data for any newly found quotes
    for element in quote_elements:
        try:
            quote_text = element.find_element(By.CLASS_NAME, "text").text

            if quote_text not in scraped_quotes:
                author = element.find_element(By.CLASS_NAME, "author").text

                tags_elements = element.find_elements(By.CLASS_NAME, "tag")
                tags_list = [t.text for t in tags_elements]

                scraped_quotes[quote_text] = {
                    'Quote text': quote_text,
                    'Author name': author,
                    'Tags': ", ".join(tags_list),
                    'Number of tags': len(tags_list),
                    'Scroll batch': current_batch
                }
        except Exception as e:
            print(f"error extracting an element: {e}")

    driver.execute_script("window.scrollTo(0, document.documentElement.scrollHeight);")

    #explicit wait to check if the number of quotes increases
    try:

        wait.until(lambda d: len(d.find_elements(By.CLASS_NAME, "quote")) > len(quote_elements))

        empty_scroll_count = 0
        current_batch += 1
        print(f"scrolled successfully. Now processing batch {current_batch}.")

    except:
        empty_scroll_count += 1
        print(f"No new content loaded. Consecutive empty scrolls: {empty_scroll_count}")

    if empty_scroll_count >= 2:
        print("Reached the absolute bottom (two consecutive empty scrolls). Done.")
        break

driver.quit()

quotes_list = list(scraped_quotes.values())
df_dynamic = pd.DataFrame(quotes_list)

csv_filename_dynamic = '23L_2503_versionB_dynamic_quotes.csv'
df_dynamic.to_csv(csv_filename_dynamic, index=False)

print(f"\nSuccess ^_^ : Extracted a total of {len(df_dynamic)} unique quotes.")
print(f"Data saved to {csv_filename_dynamic}")

df_dynamic.head()

starting dynamic scrolling..
scrolled successfully. Now processing batch 1.
scrolled successfully. Now processing batch 2.
scrolled successfully. Now processing batch 3.
scrolled successfully. Now processing batch 4.
scrolled successfully. Now processing batch 5.
scrolled successfully. Now processing batch 6.
scrolled successfully. Now processing batch 7.
scrolled successfully. Now processing batch 8.
scrolled successfully. Now processing batch 9.
No new content loaded. Consecutive empty scrolls: 1
No new content loaded. Consecutive empty scrolls: 2
Reached the absolute bottom (two consecutive empty scrolls). Done.

Success ^_^ : Extracted a total of 100 unique quotes.
Data saved to 23L_2503_versionB_dynamic_quotes.csv


,Quote text,Author name,Tags,Number of tags,Scroll batch
0,“The world as we have created it is a process ...,Albert Einstein,"change, deep-thoughts, thinking, world",4,0
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling,"abilities, choices",2,0
2,“There are only two ways to live your life. On...,Albert Einstein,"inspirational, life, live, miracle, miracles",5,0
3,"“The person, be it gentleman or lady, who has ...",Jane Austen,"aliteracy, books, classic, humor",4,0
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe,"be-yourself, inspirational",2,0


Bonus

In [3]:
!pip install playwright nest_asyncio
!playwright install
!playwright install-deps

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 25.6 MB/s eta 0:00:00
184.3 MiB [] 0% 379.7s184.3 MiB [] 0% 68.3s184.3 MiB [] 0% 42.7s184.3 MiB [] 0% 36.8s184.3 MiB [] 0% 30.7s184.3 MiB [] 0% 32.2s184.3 MiB [] 0% 28.3s184.3 MiB [] 0% 21.5s184.3 MiB [] 0% 16.2s184.3 MiB [] 1% 11.9s184.3 MiB [] 1% 9.8s184.3 MiB [] 2% 8.9s184.3 MiB [] 2% 8.8s184.3 MiB [] 2% 8.6s184.3 MiB [] 3% 8.1s184.3 MiB [] 3% 7.8s184.3 MiB [] 3% 7.7s184.3 MiB [] 3% 7.5s184.3 MiB [] 4% 7.5s184.3 MiB [] 4% 7.3s184.3 MiB [] 5% 7.3s184.3 MiB [] 5% 7.2s184.3 MiB [] 5% 7.1s184.3 MiB [] 6% 7.0s184.3 MiB [] 6% 6.9s184.3 MiB [] 6% 6.8s184.3 MiB [] 6% 7.2s184.3 MiB [] 6% 7.5s184.3 MiB [] 7% 7.6s184.3 MiB [] 7% 7.9s184.3 MiB [] 7% 8.1s184.3 MiB [] 8% 7.5s184.3 MiB [] 8% 7.1s184.3 MiB [] 9% 6.7s184.3 MiB [] 9% 6.3s184.3 MiB [] 10% 6.1s184.3 MiB [] 11% 5.9s184.3 MiB [] 11% 5.8s184.3 MiB [] 12% 5.6s184.3 MiB [] 12% 5.4s184.3 MiB [] 13% 5.3s184.3 MiB [] 13% 5.2s184.3 MiB [] 13% 5.1s184.3 MiB [] 14% 5.2s184.3 MiB [] 14% 5.1s

In [4]:
import nest_asyncio
nest_asyncio.apply()

from playwright.async_api import async_playwright
import asyncio

async def scrape_with_playwright():
    async with async_playwright() as p:

        browser = await p.chromium.launch(headless=True) #launch chromium headless browser
        page = await browser.new_page()

        print("playwright: opening page..")
        await page.goto("https://quotes.toscrape.com/scroll")

        await page.wait_for_selector('.quote')

        print("playwright: ccrolling to trigger dynamic load..")

        await page.evaluate("window.scrollTo(0, document.body.scrollHeight)")

        await page.wait_for_function("document.querySelectorAll('.quote').length > 10")

        quotes = await page.query_selector_all('.quote')
        print(f"Playwright: successfully loaded {len(quotes)} quotes after scrolling")

        print("\nsample quotes extracted via playwright:")
        for i in range(2):
            text_element = await quotes[i].query_selector('.text')
            text = await text_element.inner_text()
            print(f"- {text}")

        await browser.close()

asyncio.run(scrape_with_playwright())

playwright: opening page..
playwright: ccrolling to trigger dynamic load..
Playwright: successfully loaded 20 quotes after scrolling

sample quotes extracted via playwright:
- “The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”
- “It is our choices, Harry, that show what we truly are, far more than our abilities.”


**Scraping Methodology**



***Question 1: Static Web Scraping***

for this part i used requests and beautifulsoup. To find the elements i opened chrome developer tools and right clicked to inspect the page. At first i was clicking the wrong things and targeting the wrong div so my lists were coming up empty. It took me a while to realize i had to target the article tag with the class product_pod to actually get the books. For navigation my code looks for the li tag with the class next and extracts the link to go to the next page. The program knows it is finished when it searches for that next button and returns nothing so it just breaks the loop. to verify my data i made the scraper grab the total expected results from the page and compare it to my actual scraped count at the end of each category.  





**Question 2: Dynamic Web Scraping**

For the dynamic page i first checked with requests and found 0 quotes in the HTML which proved i needed selenium. Navigating the page meant using a javascript command to scroll to the bottom of the page. For the waits i used WebDriverWait instead of time.sleep() because the assignment required explicit waits. it waits until the count of quote elements on the page strictly increases. My program decides it is completely done when it tries to scroll and wait but no new quotes load for two consecutive times.
i faced a lot of challenges here. first i tried to run the code but got a NameError because i forgot to import the webdriver stuff. then when i added the imports i got a ModuleNotFoundError because i didn't realize i had to actually pip install selenium in colab before using it. i thought it was just already there like pandas. i fixed it by adding the install commands in a cell above. to verify the data i checked the csv to make sure the scroll batches incremented properly and there were no duplicates.  




**Bonus: Playwright**

I also tried the playwright bonus. My main challenge was setting it up because it required different install commands and i had to use nest_asyncio to make it work inside the colab environment. for the comparison playwright felt easier to wait for elements because of its auto waiting features compared to selenium.  